In [19]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.optimize import curve_fit

np.random.seed(42)

In [20]:
def truemodel(xt):
  return np.column_stack((
    2.0 - xt[:,0] + 2.0*xt[:,0]**2 - 3.0*xt[:,0]**3 + xt[:,1] + xt[:,0]*xt[:,1],
    2.0 - xt[:,0] + 2.0*xt[:,0]**2 - 3.0*xt[:,0]**3 + xt[:,1] + xt[:,0]*xt[:,1] - 1.0 + 0.5*xt[:,0] - 0.25*xt[:,1]
  ))

In [21]:
x = np.linspace(0,1,100)
y = np.linspace(0,1,100)

X, Y = np.meshgrid(x, y)
coords = np.column_stack([X.ravel(), Y.ravel()])

z_true = truemodel(coords)

In [22]:
target_1 = z_true[:,0].reshape(100,100)
target_2 = z_true[:,1].reshape(100,100)

In [23]:
true_target1 = go.Surface(z=target_1, colorscale=['rgb( 99, 110, 250)','rgb( 99, 110, 250)'], x=x, y=y, opacity=0.5, showscale=False, name='True comp')
true_target2 = go.Surface(z=target_2, colorscale=['rgb(239, 101,  72)','rgb(239, 101,  72)'], x=x, y=y, opacity=0.5, showscale=False, name='True expt')

In [24]:
comp_sample = 100
expt_sample = 20

mask = (coords[:,0] == 0.0) ^ (coords[:,1] <= 0.0)
masked_coords = coords[mask]
masked_z_true = z_true[mask]

index_comp = np.random.choice(coords.shape[0], size=comp_sample, replace=False)
index_expt = np.random.choice(masked_coords.shape[0], size=expt_sample, replace=False)
comp_descriptor = coords[index_comp]
expt_descriptor = masked_coords[index_expt]

comp_target = z_true[index_comp, 0] + 0.05*np.random.randn(comp_sample)
expt_target = masked_z_true[index_expt, 1] + 0.2*np.random.randn(expt_sample)

In [25]:
psample_comp = go.Scatter3d(x=comp_descriptor[:,0], y=comp_descriptor[:,1], z=comp_target, mode='markers', marker=dict(size=2, color='blue'), name='sample comp')
psample_expt = go.Scatter3d(x=expt_descriptor[:,0], y=expt_descriptor[:,1], z=expt_target, mode='markers', marker=dict(size=2, color='red'), name='sample expt')

In [26]:
from scipy.optimize import curve_fit

def ols_model(xt, a, b, c, d, e,f):
  return a - f*xt[0] + b*xt[0]**2 - c*xt[0]**3 + d*xt[1] + e*xt[0]*xt[1]

def ols_z(x, y, params):
  return ols_model([x, y], *params)

ols_params_target1, _ = curve_fit(ols_model, comp_descriptor.T, comp_target)

ols_target1 = ols_model(coords.T, *ols_params_target1)
ols_target1_surface = go.Surface(z=ols_z(X,Y,ols_params_target1), x=x, y=y, opacity=0.5, colorscale=['rgb( 79, 110, 250)','rgb( 79, 110, 250)'], showscale=False, name="ols comp")

ols_params_target2, _ = curve_fit(ols_model, expt_descriptor.T, expt_target)

ols_target2 = ols_model(coords.T, *ols_params_target2)
ols_target2_surface = go.Surface(z=ols_z(X,Y,ols_params_target2), x=x, y=y, opacity=0.5, colorscale=['black','black'], showscale=False, name="ols target")

/var/folders/5q/z8g_pbg51dxc9q44_yqyjj840000gn/T/ipykernel_21415/3722251731.py:14: OptimizeWarning: Covariance of the parameters could not be estimated
  ols_params_target2, _ = curve_fit(ols_model, expt_descriptor.T, expt_target)


In [29]:
x.shape

(100, 100)

In [27]:
fig = go.Figure(data=[psample_comp, psample_expt, ols_target1_surface, ols_target2_surface, true_target1, true_target2])
fig.update_layout(width=1000,height=1000)
fig.update_layout(scene_aspectmode='cube')
fig.update_layout(
    scene=dict(
        xaxis=dict(title='descriptor1'),
        yaxis=dict(title='descriptor2'),
        zaxis=dict(title='target')
    )
)
fig.update_layout(
    title={
        'text': r'toy model',
        'x': 0.5,
        'y': 0.9,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=30)
    }
)
fig.update_layout(legend=dict(font=dict(size=20)))

fig.update_layout(
    legend=go.layout.Legend(
        itemsizing='constant'
    )
)

fig.show()

In [10]:
descriptor = np.vstack((comp_descriptor, expt_descriptor))
target = np.zeros_like(descriptor)
target[:,:] = np.nan
target[:comp_sample, 0] = comp_target
target[comp_sample:, 1] = expt_target

In [11]:
def truemodel(xt):
  return np.column_stack((
    2.0 - xt[:,0] + 2.0*xt[:,0]**2 - 3.0*xt[:,0]**3 + xt[:,1] + xt[:,0]*xt[:,1],
    2.0 - xt[:,0] + 2.0*xt[:,0]**2 - 3.0*xt[:,0]**3 + xt[:,1] + xt[:,0]*xt[:,1] - 1.0 + 0.5*xt[:,0] - 0.25*xt[:,1]
  ))

In [12]:
import CLAUDE as claude

da = claude.CLAUDE(nDescriptor = 2, \
                   nTarget = 2, \
                   degreePolynomialDescriptor = np.array([[0,0],[1,0],[2,0],[3,0],[0,1],[1,1]]), \
                   degreeActiveDescriptor = [
                     [[0,0],[0,1]],
                     [[0,1],[0,1]],
                     [[1,0],[0,1]]
                    ], # yc is default to be always true
                    # we know that that target - expt = const + x0 + x1
                    # thus we activate 0,0 (const) 0,1 (x0) 1,0 (x1) for ye (0,1)
                   minimizationMethod="SLSQP")

da.fit(descriptor,target)

aq = da.acquisitionFunction(coords)
idxMax1 = np.argmax(coords[:,0]) #なぜ定義するのか？
idxMax2 = np.argmax(coords[:,1])

da.saveModelParameterToFile()
da.saveCovarianceMatrixToFile()

/Users/syahrizailm/Codes/lab/CLAUDE/da_tutorial_2d/CLAUDE.py:1053: OptimizeWarning:

Unknown solver options: gtol

/Users/syahrizailm/Codes/lab/CLAUDE/da_tutorial_2d/CLAUDE.py:1215: RuntimeWarning:

invalid value encountered in log



In [13]:
da_res = da.predict(coords)
da_target1 = go.Surface(z=da_res[:,0].reshape(100,100), colorscale=['rgb( 99, 110, 250)','rgb( 99, 110, 250)'], x=x, y=y, opacity=0.5, showscale=False)
da_target2 = go.Surface(z=da_res[:,1].reshape(100,100), colorscale=["green", "green"], x=x, y=y, opacity=0.5, showscale=False)

In [14]:
fig = go.Figure(data=[da_target1, da_target2, true_target1, true_target2, psample_comp, psample_expt, ols_target1_surface, ols_target2_surface])
fig.update_layout(width=800,height=500)
fig.update_layout(scene_aspectmode='cube')
fig.update_layout(
    scene=dict(
        xaxis=dict(title='descriptor1'),
        yaxis=dict(title='descriptor2'),
        zaxis=dict(title='target')
    )
)
fig.update_layout(
    title={
        'text': r'toy model',
        'x': 0.5,
        'y': 0.9,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=30)
    }
)
fig.update_layout(legend=dict(font=dict(size=20)))

fig.update_layout(
    legend=go.layout.Legend(
        itemsizing='constant'
    )
)

fig.show()